In [1]:
import torch
from torch import nn
from d2l import torch as d2l
from good_functions import *

batch_size=256
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)

softmax回归的输出层是一个全连接层

In [2]:
net=nn.Sequential(nn.Flatten(),nn.Linear(784,10)) #把图像变成向量

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,std=0.01)

net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)

## 模型定义与初始化解析

### 逐行解析

- **`nn.Sequential(...)`**：按顺序串联多个层的容器，数据从第一层流到最后一层。
  - `nn.Flatten()`：展平图像 `(batch, 1, 28, 28)` → `(batch, 784)`，等价于手写的 `X.reshape((-1, 784))`。
  - `nn.Linear(784, 10)`：全连接层，执行 `Y = XW + b`，等价于手写的 `matmul(X, w) + b`。
    - 输入 784 = 28×28 像素数，输出 10 = 类别数。
    - 内部自动创建权重 `W(784,10)` 和偏置 `b(10,)`，不需要手动定义。

- **`def init_weights(m)`**：权重初始化函数。
  - `m` 是模型中的某一层，`apply` 会逐层传进来。
  - `if type(m) == nn.Linear`：只处理 Linear 层，跳过 Flatten。
  - `nn.init.normal_(m.weight, std=0.01)`：用正态分布（均值 0，标准差 0.01）覆盖默认权重，等价于手写的 `torch.normal(0, 0.01, ...)`。带 `_` 后缀 = 原地操作。

- **`net.apply(init_weights)`**：递归遍历 net 的每一层，调用 `init_weights`。
  - `init_weights(nn.Flatten())` → 不是 Linear，跳过。
  - `init_weights(nn.Linear(784,10))` → 是 Linear，初始化权重。

### 与手写版的对照

| 手写版 | API 版 | 对应关系 |
|--------|--------|---------|
| `w = torch.normal(0, 0.01, (784,10))` | `nn.Linear` + `init_weights` | 权重参数 |
| `b = torch.zeros(10)` | `nn.Linear` 自带偏置 | 偏置参数 |
| `X.reshape((-1, 784))` | `nn.Flatten()` | 展平图像 |
| `softmax(matmul(X,w)+b)` | `nn.Sequential` 输出 logits | 前向传播 |

**关键点**
- API 版不把 `softmax` 写在模型里——训练时用 `nn.CrossEntropyLoss`（内部自带 softmax），预测时再单独加。
- `nn.Linear` 自带 `requires_grad=True`，配合 `optim.SGD(net.parameters(), lr)` 自动获取所有参数，不再需要手动传 `[w, b]`。

在交叉熵损失函数中传递未归一化的预测，同时计算softmax及其对数

In [3]:
loss=nn.CrossEntropyLoss()

使用学习率未0.1的小批量随机梯度下降的优化算法

In [4]:
trainer=torch.optim.SGD(net.parameters(),lr=0.1)

## torch.optim.SGD：创建优化器

```python
trainer = torch.optim.SGD(net.parameters(), lr=0.1)
```

### 逐部分解析

- **`net.parameters()`**：自动提取模型中所有需要学习的参数（`W` 和 `b`），不需要手动传 `[w, b]`。
- **`torch.optim.SGD`**：随机梯度下降优化器，负责在反向传播后更新参数。
- **`lr=0.1`**：学习率，控制每次更新的步长。太大不收敛，太小收敛慢。

### 做了什么

这一行创建了优化器对象 `trainer`，后续训练循环中会调用：
1. `trainer.zero_grad()` — 清空旧梯度
2. `loss.backward()` — 反向传播计算新梯度
3. `trainer.step()` — 用 SGD 公式更新参数：`param -= lr * param.grad`

### 与手写版的对照

| 手写版 | API 版 | 对应关系 |
|--------|--------|---------|
| `def sgd(params, lr, batch_size)` | `torch.optim.SGD` | 优化算法 |
| `def updater(batch_size)` | `trainer` 对象 | 更新器 |
| `d2l.sgd([w,b], lr, batch_size)` | `trainer.step()` | 执行更新 |
| 手动 `param.grad.zero_()` | `trainer.zero_grad()` | 清空梯度 |
| 手动传 `[w, b]` | `net.parameters()` | 自动获取参数 |

**关键点**
- API 版的 `train_epoch_ch3` 会走 `isinstance(updater, torch.optim.Optimizer)` 的 **if 分支**（因为 `trainer` 是 Optimizer 实例），用 `l.mean().backward()` + `trainer.step()`。
- 手写版走 **else 分支**，用 `l.sum().backward()` + `updater(batch_size)`。两条路径数学上等价。

In [5]:

num_epochs=10
train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer)

epoch 1, loss 0.7861, train acc 0.7509, test acc 0.7853
epoch 2, loss 0.5693, train acc 0.8128, test acc 0.8079
epoch 3, loss 0.5263, train acc 0.8253, test acc 0.8182
epoch 4, loss 0.5002, train acc 0.8320, test acc 0.8094
epoch 5, loss 0.4852, train acc 0.8367, test acc 0.8235
epoch 6, loss 0.4740, train acc 0.8399, test acc 0.8242
epoch 7, loss 0.4648, train acc 0.8434, test acc 0.8251
epoch 8, loss 0.4578, train acc 0.8455, test acc 0.8301
epoch 9, loss 0.4525, train acc 0.8466, test acc 0.8292
epoch 10, loss 0.4476, train acc 0.8485, test acc 0.7871


(0.44755092617670694, 0.8484833333333334)